In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class Perceptron:
    def __init__(self, learning_rate=0.1, n_epochs=50, random_state=42):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.w_ = None
        self.b_ = None
        self.errors_ = []

    def _activation(self, z):
        return (z >= 0).astype(int)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)

        rng = np.random.RandomState(self.random_state)
        self.w_ = rng.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b_ = 0.0
        self.errors_ = []

        for _ in range(self.n_epochs):
            errors = 0
            for xi, target in zip(X, y):
                z = np.dot(xi, self.w_) + self.b_
                y_pred = int(self._activation(z))
                update = self.learning_rate * (target - y_pred)

                self.w_ += update * xi
                self.b_ += update

                errors += int(update != 0.0)
            self.errors_.append(errors)

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        z = X @ self.w_ + self.b_ #матричное умножение с весами
        return self._activation(z).astype(int)

In [ ]:
X_and = np.array([
    [0, 0], [0, 1], [1, 0], [1, 1]
], dtype=float)

y_and = np.array([0, 0, 0, 1], dtype=int)

X_or = X_and.copy()
y_or = np.array([0, 1, 1, 1], dtype=int)

X_xor = X_and.copy()
y_xor = np.array([0, 1, 1, 0], dtype=int)

In [ ]:
p_and = Perceptron(learning_rate=0.1, n_epochs=50, random_state=42).fit(X_and, y_and)
pred_and = p_and.predict(X_and)

print("AND: w =", p_and.w_, " b =", p_and.b_)
print("AND предсказания:", pred_and)
print("AND:", y_and)
print("ошибки_:", p_and.errors_)

AND: w = [0.10496714 0.09861736]  b = -0.2
AND предсказания: [0 0 0 1]
AND: [0 0 0 1]
ошибки_: [2, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Сделать многослойный перцептрон

In [1]:
class MultiLayerPerceptron:
    def __init__(self, learning_rate=0.1, n_epochs=5000, n_hidden=2, random_state=42):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.n_hidden = n_hidden
        self.random_state = random_state


        self.W1_ = None
        self.b1_ = None
        self.W2_ = None
        self.b2_ = None

        self.losses_ = []

    def _sigmoid(self, z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def _sigmoid_deriv_from_a(self, a):
        return a * (1.0 - a)

    def _activation(self, z):
        return (z >= 0.5).astype(int)

    def _forward(self, X):
        # скрытый слой
        z1 = X @ self.W1_ + self.b1_
        a1 = self._sigmoid(z1)

        # выходной слой
        z2 = a1 @ self.W2_ + self.b2_
        a2 = self._sigmoid(z2)

        cache = (X, z1, a1, z2, a2)
        return a2, cache

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)

        y = y.reshape(-1, 1)

        rng = np.random.RandomState(self.random_state)
        n_features = X.shape[1]

        self.W1_ = rng.normal(loc=0.0, scale=0.1, size=(n_features, self.n_hidden))
        self.b1_ = np.zeros(self.n_hidden, dtype=float)
        self.W2_ = rng.normal(loc=0.0, scale=0.1, size=(self.n_hidden, 1))
        self.b2_ = np.zeros(1, dtype=float)

        self.losses_ = []

        for _ in range(self.n_epochs):
            a2, (Xc, z1, a1, z2, a2c) = self._forward(X)

            eps = 1e-12
            loss = -np.mean(y * np.log(a2 + eps) + (1 - y) * np.log(1 - a2 + eps))
            self.losses_.append(loss)


            dZ2 = (a2 - y)
            dW2 = (a1.T @ dZ2) / X.shape[0]
            db2 = np.mean(dZ2, axis=0)

            dA1 = dZ2 @ self.W2_.T
            dZ1 = dA1 * self._sigmoid_deriv_from_a(a1)
            dW1 = (X.T @ dZ1) / X.shape[0]
            db1 = np.mean(dZ1, axis=0)

            lr = self.learning_rate
            self.W2_ -= lr * dW2
            self.b2_ -= lr * db2
            self.W1_ -= lr * dW1
            self.b1_ -= lr * db1

        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        a2, _ = self._forward(X)
        return a2.ravel()

    def predict(self, X):
        proba = self.predict_proba(X)
        return self._activation(proba).astype(int)

In [4]:
import numpy as np

X_xor = np.array([
    [0,0],
    [0,1],
    [1,0],
    [1,1]
], dtype=float)

y_xor = np.array([0,1,1,0], dtype=int)

In [9]:
mlp = MultiLayerPerceptron(
    learning_rate=0.7,
    n_epochs=5000,
    n_hidden=2,
    random_state=42
)

mlp.fit(X_xor, y_xor)

In [10]:
pred = mlp.predict(X_xor)

print("X:")
print(X_xor)

print("True XOR:")
print(y_xor)

print("Predicted:")
print(pred)

X:
[[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]
True XOR:
[0 1 1 0]
Predicted:
[1 0 1 0]
